In [3]:
import os
for root, _, files in os.walk('/kaggle/input'):
    print(root, files)

/kaggle/input []
/kaggle/input/competitions []
/kaggle/input/competitions/GiveMeSomeCredit ['Data Dictionary.xls', 'cs-training.csv', 'sampleEntry.csv', 'cs-test.csv']


In [4]:
import pandas as pd

df = pd.read_csv('/kaggle/input/competitions/GiveMeSomeCredit/cs-training.csv', index_col=0)
print(df.shape)
print(df.isnull().sum())
print(df['SeriousDlqin2yrs'].value_counts(normalize=True))


(150000, 11)
SeriousDlqin2yrs                            0
RevolvingUtilizationOfUnsecuredLines        0
age                                         0
NumberOfTime30-59DaysPastDueNotWorse        0
DebtRatio                                   0
MonthlyIncome                           29731
NumberOfOpenCreditLinesAndLoans             0
NumberOfTimes90DaysLate                     0
NumberRealEstateLoansOrLines                0
NumberOfTime60-89DaysPastDueNotWorse        0
NumberOfDependents                       3924
dtype: int64
SeriousDlqin2yrs
0    0.93316
1    0.06684
Name: proportion, dtype: float64


In [5]:
import numpy as np
from sklearn.model_selection import train_test_split

df = df.rename(columns={'SeriousDlqin2yrs': 'default'})

df['income_missing'] = df['MonthlyIncome'].isnull().astype(int)
df['MonthlyIncome'] = df['MonthlyIncome'].fillna(df['MonthlyIncome'].median())
df['NumberOfDependents'] = df['NumberOfDependents'].fillna(0)

df = df[df['age'] > 18]

for col in ['RevolvingUtilizationOfUnsecuredLines', 'DebtRatio', 'MonthlyIncome']:
    df[col] = df[col].clip(upper=df[col].quantile(0.999))

for col in ['NumberOfTime30-59DaysPastDueNotWorse',
            'NumberOfTimes90DaysLate',
            'NumberOfTime60-89DaysPastDueNotWorse']:
    df[col] = df[col].clip(upper=20)

X = df.drop('default', axis=1)
y = df['default']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)
print(df.isnull().sum().sum(), "missing values left")

(119999, 11) (30000, 11)
0 missing values left


In [6]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

ratio = (y_train == 0).sum() / (y_train == 1).sum()

models = {
    'Logistic Regression': Pipeline([
        ('scale', StandardScaler()),
        ('model', LogisticRegression(max_iter=1000, class_weight='balanced'))
    ]),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, min_samples_leaf=5, class_weight='balanced',
        n_jobs=-1, random_state=42
    ),
    'XGBoost': XGBClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        scale_pos_weight=ratio, eval_metric='auc',
        n_jobs=-1, random_state=42
    ),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ['roc_auc', 'average_precision', 'recall', 'precision']

rows = []
for name, m in models.items():
    s = cross_validate(m, X_train, y_train, cv=cv, scoring=scoring, n_jobs=1)
    rows.append({
        'Model': name,
        'ROC-AUC': s['test_roc_auc'].mean(),
        'AUC std': s['test_roc_auc'].std(),
        'PR-AUC': s['test_average_precision'].mean(),
        'Recall': s['test_recall'].mean(),
        'Precision': s['test_precision'].mean(),
    })

print(pd.DataFrame(rows).round(3).to_string(index=False))

              Model  ROC-AUC  AUC std  PR-AUC  Recall  Precision
Logistic Regression    0.816    0.006   0.345   0.622      0.244
      Random Forest    0.855    0.005   0.384   0.490      0.390
            XGBoost    0.864    0.006   0.400   0.766      0.217


In [7]:
import optuna
from sklearn.model_selection import cross_val_score

optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 600),
        'max_depth': trial.suggest_int('max_depth', 3, 7),
        'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.15, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.5, 10, log=True),
    }
    m = XGBClassifier(**params, scale_pos_weight=ratio, eval_metric='auc',
                      n_jobs=-1, random_state=42)
    cv3 = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    return cross_val_score(m, X_train, y_train, cv=cv3, scoring='roc_auc').mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20)

print("Best ROC-AUC:", round(study.best_value, 4))
print(study.best_params)

Best ROC-AUC: 0.8649
{'n_estimators': 544, 'max_depth': 3, 'learning_rate': 0.023909252654715864, 'subsample': 0.8211851663142686, 'colsample_bytree': 0.6367934335804545, 'min_child_weight': 1, 'reg_lambda': 1.1114345710337308}
